# 03 — Nihai Model, Kilitli Test Tahminleri ve Bağımsız Test

**Yapay Sinir Ağları — Yüksek Lisans Ara Sınav Projesi** · Hakan Kızılhan

Deneysel akışın (yönerge Bölüm 5) 9–10. adımları. Kısaltmalar: MLP (Multi-Layer Perceptron), AP (Average Precision, ortalama kesinlik), ROC-AUC (Receiver Operating Characteristic – Area Under Curve), PR (Precision–Recall), CI (Confidence Interval, güven aralığı), FP/FN (False Positive/False Negative, yanlış pozitif/negatif), SD (Standard Deviation).

| Bölüm | İşlem |
|---|---|
| A | Nihai konfigürasyonun (02'de seçilen M5) tüm development verisinde 3 önceden ilan edilmiş seed ile eğitimi; Logistic Regression referansı; model, ön işleme durumu ve `final_hyperparameters.json` kaydı; test **özelliklerinden** etiketsiz tahminler → `test_predictions_locked.csv` |
| ⛔ | Kilitli tahminler ayrı bir commit ile GitHub'a yüklenir; aşağıdaki kontrol hücresi bu commit yapılmadan test etiketlerinin okunmasına izin vermez |
| B | Test etiketleri yalnızca bu noktada okunur: metrikler, confusion matrix, ROC/PR eğrileri, bootstrap %95 CI, seed kararlılığı, hata analizi |

**Önceden ilan edilen ana model:** nihai konfigürasyon + birincil ağırlık seed'i **31065**. Diğer iki seed (32065, 33065) yalnızca kararlılık analizi içindir; en yüksek test skoru ana sonuç olarak seçilmez. Sınıflandırma eşiği **0.50** (eşik araması yapılmadı). Test sonuçlarına bakılarak modele, eşiğe veya ön işlemeye müdahale edilmez.

In [ ]:
import hashlib
import json
import pickle
import platform
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
from matplotlib.patches import FancyBboxPatch
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, precision_recall_curve, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline

import ysa_core as yc

ROOT = Path.cwd()
RES_DIR, FIG_DIR, MODEL_DIR = ROOT / "results", ROOT / "figures", ROOT / "results" / "models"
MODEL_DIR.mkdir(exist_ok=True)
cfgs = yc.load_configs(ROOT)
TRAIN, SEEDS = cfgs["training"], cfgs["seeds"]
yc.set_global_seed(SEEDS["global_python_numpy_torch"])
torch.set_num_threads(1)
yc.set_plot_style()
pd.set_option("display.max_columns", 50, "display.width", 180)

SELECTION = yc.load_json(RES_DIR / "final_selection.json")
FINAL_CFG = SELECTION["config"]
THR = SELECTION["threshold"]
SEED_LIST, PRIMARY = SEEDS["weight_init_stability"], SEEDS["weight_init_primary"]
assert SEED_LIST[0] == PRIMARY

X, y = yc.load_development(ROOT, TRAIN["data"])
print(f"Nihai konfigürasyon: {SELECTION['selected_candidate']} ({SELECTION['selected_search']} arama) → "
      f"{yc.describe_candidate(FINAL_CFG)}")
print(f"Development iç CV AP: {SELECTION['dev_inner_AP_mean']:.4f} ± {SELECTION['dev_inner_AP_sd']:.4f} | "
      f"seed'ler: {SEED_LIST} (birincil {PRIMARY}) | eşik: {THR}")


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")
    print(f"kaydedildi: figures/{name}.png")

## Bölüm A1 — Nihai eğitim (3 seed) ve referans model

Her seed için: development verisi %85 fit / %15 stratified e-stop alt kümesine ayrılır (bölünme tüm seed'lerde aynıdır; yalnızca ağırlık başlangıcı ve mini-batch sırası değişir), ön işleme fit alt kümesinde öğrenilir, early stopping ve ReduceLROnPlateau e-stop kaybını izler. Logistic Regression, outer CV'deki sabit kuralla tüm development verisinde eğitilir.

In [ ]:
finals, final_rows = {}, []
for s in SEED_LIST:
    fit = yc.fit_candidate(FINAL_CFG, X, y, cfgs, weight_seed=s)
    finals[s] = fit
    final_rows.append({"seed": s, "primary": s == PRIMARY, **fit.info})
    print(f"seed {s}: en iyi epoch {fit.info['best_epoch']}, durulan {fit.info['stopped_epoch']}, "
          f"LR düşüşü {fit.info['n_lr_drops']}, parametre {fit.info['n_params']}, {fit.info['total_time_s']:.1f} sn, "
          f"uyarı: {fit.info['warnings'] or 'yok'}")

final_runs = pd.DataFrame(final_rows)
final_runs.to_csv(RES_DIR / "final_training_runs.csv", index=False)
pd.concat([f.history.assign(seed=s) for s, f in finals.items()]).to_csv(RES_DIR / "final_training_history.csv",
                                                                         index=False)

logreg = Pipeline([("pre", yc.reference_preprocessor(X)), ("clf", LogisticRegression(max_iter=5000))]).fit(X, y)
print("Logistic Regression eğitildi (tüm development, sabit ön işleme, C=1.0)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 5.8), sharex="col", gridspec_kw={"height_ratios": [2, 1]})
for j, (s, fit) in enumerate(finals.items()):
    h, ax, axl = fit.history, axes[0, j], axes[1, j]
    ax.plot(h["epoch"], h["train_loss"], color=yc.METHOD_COLORS["MLP_manual"], lw=2, label="train (fit) loss")
    ax.plot(h["epoch"], h["es_loss"], color=yc.METHOD_COLORS["LogReg"], lw=2, label="e-stop loss")
    ax.axvline(fit.info["best_epoch"], color=yc.INK, ls="--", lw=1)
    ax.text(fit.info["best_epoch"], ax.get_ylim()[1], f" en iyi: {fit.info['best_epoch']}", va="top", fontsize=9)
    ax.set_title(f"seed {s}" + (" (birincil)" if s == PRIMARY else ""), fontsize=11)
    ax.set_ylabel("BCE" if j == 0 else "")
    axl.step(h["epoch"], h["lr"], where="post", color=yc.METHOD_COLORS["MLP_random"], lw=2)
    axl.set_yscale("log")
    axl.set_xlabel("epoch")
    axl.set_ylabel("LR" if j == 0 else "")
axes[0, 0].legend(fontsize=9)
fig.suptitle(f"Nihai model ({SELECTION['selected_candidate']}) — eğitim ve e-stop kaybı, epoch bazlı LR",
             fontweight="bold", y=1.01)
fig.tight_layout()
savefig(fig, "final_training_curves")
plt.show()

## Bölüm A2 — MLP mimarisi ve öğrenilebilir parametre sayısı (Şekil 3)

Her `Linear(n_in → n_out)` katmanı `n_in × n_out` ağırlık + `n_out` bias içerir. ReLU ve sigmoid parametresizdir. Girdi boyutu, fit alt kümesinde öğrenilen OHE kategorileri ve `pdays` göstergesine göre belirlenir.

In [ ]:
model = finals[PRIMARY].model
linears = [m for m in model.net if isinstance(m, torch.nn.Linear)]
layer_rows = [{"katman": f"Linear {l.in_features}→{l.out_features}", "ağırlık": l.in_features * l.out_features,
               "bias": l.out_features, "toplam": l.in_features * l.out_features + l.out_features} for l in linears]
layer_table = pd.DataFrame(layer_rows)
layer_table.loc[len(layer_table)] = ["TOPLAM", layer_table["ağırlık"].sum(), layer_table["bias"].sum(),
                                     layer_table["toplam"].sum()]
assert layer_table["toplam"].iloc[-1] == yc.count_params(model)

fig, ax = plt.subplots(figsize=(13, 2.5))
ax.set_xlim(0, 13)
ax.set_ylim(0.7, 3.15)
ax.axis("off")
act = FINAL_CFG["activation"]
blocks = [(f"Girdi\n{linears[0].in_features} özellik", "#e6e5e0", "")]
blocks += [(f"Linear {l.in_features}→{l.out_features}\n+ {act}", "#cde2fb", f"{r['toplam']:,} param".replace(",", "."))
           for l, r in zip(linears[:-1], layer_rows[:-1])]
blocks += [(f"Linear {linears[-1].in_features}→1\n(logit)", "#cde2fb", f"{layer_rows[-1]['toplam']} param"),
           ("sigmoid\nP(y = yes)", "#fbd8c8", ""), (f"eşik {THR}\n→ sınıf", "#e6e5e0", "")]
xs = np.linspace(0.9, 12.1, len(blocks))
for i, (x0, (txt, col, sub)) in enumerate(zip(xs, blocks)):
    ax.add_patch(FancyBboxPatch((x0 - 0.85, 1.25), 1.7, 1.2, boxstyle="round,pad=0.05", fc=col, ec="#8f8d86"))
    ax.text(x0, 1.85, txt, ha="center", va="center", fontsize=10)
    if sub:
        ax.text(x0, 0.95, sub, ha="center", va="center", fontsize=9, color=yc.INK)
    if i < len(blocks) - 1:
        ax.annotate("", xy=(xs[i + 1] - 0.9, 1.85), xytext=(x0 + 0.9, 1.85),
                    arrowprops=dict(arrowstyle="->", color=yc.INK, lw=1.4))
ax.text(6.5, 2.9, f"Nihai MLP ({SELECTION['selected_candidate']}): toplam {yc.count_params(model):,} öğrenilebilir parametre "
        f"· kayıp: BCEWithLogitsLoss · optimizer: {yc.OPTIMIZER_NAMES[FINAL_CFG['optimizer']]}".replace(",", "."),
        ha="center", fontsize=11, fontweight="bold")
savefig(fig, "mlp_architecture")
plt.show()
layer_table

## Bölüm A3 — Model, ön işleme durumu ve nihai hiperparametrelerin kaydı

In [ ]:
def preprocessing_state(pre):
    num_pipe, cat_pipe = pre.named_transformers_["num"], pre.named_transformers_["cat"]
    num_cols = list(pre.transformers_[0][2])
    imp, scaler = num_pipe.named_steps["impute"], num_pipe.named_steps["scale"]
    state = {"numeric_columns": num_cols,
             "numeric_median": dict(zip(num_cols, imp.statistics_.round(6).tolist())),
             "missing_indicator_for": [num_cols[i] for i in imp.indicator_.features_],
             "scaler": type(scaler).__name__,
             "scaler_params": {k: np.round(getattr(scaler, k), 6).tolist()
                               for k in ["mean_", "scale_", "center_", "min_", "data_min_", "data_max_"]
                               if hasattr(scaler, k)},
             "clipping": None if num_pipe.named_steps["clip"] == "passthrough" else {
                 "lower": num_pipe.named_steps["clip"].lower_.tolist(), "upper": num_pipe.named_steps["clip"].upper_.tolist()},
             "ohe_categories": {c: cats.tolist() for c, cats in
                                zip(pre.transformers_[1][2], cat_pipe.named_steps["ohe"].categories_)},
             "feature_names_out": pre.get_feature_names_out().tolist()}
    return state


for s, fit in finals.items():
    torch.save(fit.model.state_dict(), MODEL_DIR / f"final_mlp_seed{s}.pt")
    with open(MODEL_DIR / f"final_preprocessor_seed{s}.pkl", "wb") as fh:
        pickle.dump(fit.preprocessor, fh)
with open(MODEL_DIR / "final_logreg.pkl", "wb") as fh:
    pickle.dump(logreg, fh)

final_hp = {
    "selected_candidate": SELECTION["selected_candidate"], "selected_search": SELECTION["selected_search"],
    "selection_rule": SELECTION["rule"], "dev_inner_AP_mean": SELECTION["dev_inner_AP_mean"],
    "dev_inner_AP_sd": SELECTION["dev_inner_AP_sd"],
    "config": FINAL_CFG, "architecture": [r["katman"] for r in layer_rows], "n_params": yc.count_params(model),
    "loss": "BCEWithLogitsLoss", "threshold": THR,
    "primary_model": f"MLP seed {PRIMARY}", "weight_seeds": SEED_LIST,
    "early_stopping": TRAIN["early_stopping"], "lr_scheduler": TRAIN["lr_scheduler"],
    "early_stop_subset": {"fraction": TRAIN["cv"]["early_stop_fraction"], "seed": SEEDS["early_stop_subset"]},
    "per_seed": final_runs[["seed", "best_epoch", "stopped_epoch", "n_lr_drops", "final_lr", "best_es_loss",
                            "n_features", "n_fit", "n_es", "warnings"]].to_dict(orient="records"),
    "preprocessing_state_primary": preprocessing_state(finals[PRIMARY].preprocessor),
    "reference_model": "LogisticRegression(C=1.0, lbfgs, max_iter=5000) + M1 ile aynı sabit ön işleme",
    "environment": {"python": platform.python_version(), "torch": torch.__version__, "sklearn": sklearn.__version__,
                    "numpy": np.__version__, "pandas": pd.__version__, "platform": platform.platform(),
                    "processor": "AMD Ryzen 5 5500U (CPU, torch.set_num_threads(1))"},
}
(RES_DIR / "final_hyperparameters.json").write_text(json.dumps(final_hp, indent=2, ensure_ascii=False), encoding="utf-8")
print("kaydedildi: results/final_hyperparameters.json ve results/models/ (3 MLP + 3 ön işleyici + LogReg)")

## Bölüm A4 — Kilitli test tahminleri (etiketsiz)

Test verisi **yalnızca özellik sütunlarıyla** yüklenir (`load_test_features` hedef sütununu düşürür). Üç MLP seed'i ve Logistic Regression için olasılıklar ve 0.50 eşiğindeki sınıf tahminleri `results/test_predictions_locked.csv` dosyasına yazılır. Dosyada etiket sütunu yoktur.

In [ ]:
X_test = yc.load_test_features(ROOT, TRAIN["data"])
assert TRAIN["data"]["target"] not in X_test.columns and len(X_test) == 824

locked = pd.DataFrame({"orig_index": X_test.index})
for s, fit in finals.items():
    p = fit.predict_proba(X_test)
    locked[f"p_mlp_seed{s}"], locked[f"pred_mlp_seed{s}"] = p.round(8), (p >= THR).astype(int)
p_lr = logreg.predict_proba(X_test)[:, 1]
locked["p_logreg"], locked["pred_logreg"] = p_lr.round(8), (p_lr >= THR).astype(int)
locked.to_csv(RES_DIR / "test_predictions_locked.csv", index=False)

sha = hashlib.sha256((RES_DIR / "test_predictions_locked.csv").read_bytes()).hexdigest()
print("kaydedildi: results/test_predictions_locked.csv | sütunlar:", list(locked.columns))
print("SHA-256:", sha)
print("Tahmin edilen pozitif sayısı:", {c: int(locked[c].sum()) for c in locked.columns if c.startswith("pred")})

---
### ⛔ Commit noktası — test kilidi

Bu noktada `results/test_predictions_locked.csv`, `results/final_hyperparameters.json` ve `results/models/` dosyaları (ve kod sürümü) **ayrı bir commit** ile GitHub'a yüklenmelidir. Aşağıdaki hücre kilitli tahmin dosyasının commit'lenmiş ve sonrasında değiştirilmemiş olduğunu doğrulamadan test etiketlerini okumaz. Notebook baştan çalıştırıldığında Bölüm A aynı tahminleri deterministik olarak yeniden üretir; dosya bir bayt bile değişirse kontrol başarısız olur.

Kamusal veri setinde test etiketleri erişilebilir olduğundan bu işlem kör test garantisi değil, **zaman sıralı deney kaydıdır** (yönerge Bölüm 15).

In [ ]:
def git(*args):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd=ROOT)


LOCKED = ["results/test_predictions_locked.csv", "results/final_hyperparameters.json"]
if git("rev-parse", "--is-inside-work-tree").returncode != 0:
    print("UYARI: git repository'si bulunamadı — kilit kontrolü atlandı")
else:
    for path in LOCKED:
        last = git("log", "-1", "--format=%h %ad %s", "--date=iso", "--", path).stdout.strip()
        dirty = git("status", "--porcelain", "--", path).stdout.strip()
        assert last and not dirty, f"Önce {path} commit'lenmeli; test etiketleri ancak ondan sonra okunur."
        print(f"{path}: {last} ✓")

## Bölüm B1 — Test etiketlerinin okunması ve ana test metrikleri (Tablo 6)

Ana sonuç: önceden ilan edilmiş birincil MLP (seed 31065) ve Logistic Regression. Bootstrap: 1.000 yeniden örnekleme (seed 31071), her örneklemde iki sınıfın da bulunduğu kontrol edilir; %95 yüzdelik güven aralığı. Ek olarak MLP − LR farkı için eşleştirilmiş (aynı yeniden örneklemler) bootstrap aralığı verilir.

In [ ]:
y_test = yc.load_test_labels(ROOT, TRAIN["data"])
assert (y_test.index == locked["orig_index"]).all()
y_true = y_test.to_numpy()
print(f"Test: {len(y_true)} gözlem | pozitif: {y_true.sum()} (%{100 * y_true.mean():.2f})")

P = {"MLP": locked[f"p_mlp_seed{PRIMARY}"].to_numpy(), "LogReg": locked["p_logreg"].to_numpy()}
METRICS = ["AP", "ROC_AUC", "F1", "Sensitivity", "Specificity", "Balanced_Accuracy", "Precision", "Accuracy"]
test_metrics = pd.DataFrame({name: {**yc.classification_metrics(y_true, p, THR), "Brier": brier_score_loss(y_true, p)}
                             for name, p in P.items()}).T

B = 1000
rng = np.random.default_rng(SEEDS["bootstrap"])
boot, skipped = {k: [] for k in ["MLP", "LogReg", "Δ"]}, 0
while len(boot["MLP"]) < B:
    idx = rng.integers(0, len(y_true), len(y_true))
    yb = y_true[idx]
    if yb.min() == yb.max():
        skipped += 1
        continue
    vals = {}
    for name, p in P.items():
        pb = p[idx]
        vals[name] = (roc_auc_score(yb, pb), average_precision_score(yb, pb), f1_score(yb, (pb >= THR).astype(int), zero_division=0))
        boot[name].append(vals[name])
    boot["Δ"].append(tuple(a - b for a, b in zip(vals["MLP"], vals["LogReg"])))
print(f"Bootstrap: {B} geçerli örneklem, tek sınıflı olduğu için atlanan: {skipped}")

ci_rows = []
for name in ["MLP", "LogReg", "Δ"]:
    arr = np.array(boot[name])
    for j, met in enumerate(["ROC_AUC", "AP", "F1"]):
        point = (test_metrics.loc["MLP", met] - test_metrics.loc["LogReg", met]) if name == "Δ" else test_metrics.loc[name, met]
        lo, hi = np.percentile(arr[:, j], [2.5, 97.5])
        ci_rows.append({"model": "MLP − LogReg" if name == "Δ" else name, "metrik": met, "nokta": point,
                        "CI_alt": lo, "CI_üst": hi})
ci = pd.DataFrame(ci_rows)
ci.to_csv(RES_DIR / "bootstrap_ci.csv", index=False)

table6 = test_metrics[[*METRICS, "Brier", "TP", "FP", "TN", "FN"]].copy()
for met in ["ROC_AUC", "AP", "F1"]:
    for name in P:
        r = ci[(ci["model"] == name) & (ci["metrik"] == met)].iloc[0]
        table6.loc[name, f"{met} %95 CI"] = f"[{r['CI_alt']:.3f}, {r['CI_üst']:.3f}]"
table6.index = [f"MLP {SELECTION['selected_candidate']} (seed {PRIMARY})", "Logistic Regression"]
table6.to_csv(RES_DIR / "test_metrics.csv")
table6.round(4).T

In [ ]:
ci.round(4)

## Bölüm B2 — Seed kararlılığı (Tablo 7)

Aynı nihai konfigürasyonun üç başlangıç seed'iyle test performansı. Betimseldir: ana sonuç önceden ilan edilen birincil seed'dir.

In [ ]:
seed_rows = [{"seed": s, "birincil": s == PRIMARY,
              **yc.classification_metrics(y_true, locked[f"p_mlp_seed{s}"].to_numpy(), THR)} for s in SEED_LIST]
seed_df = pd.DataFrame(seed_rows).set_index("seed")
seed_df.to_csv(RES_DIR / "seed_stability_results.csv")
table7 = seed_df[["birincil", *METRICS]].copy()
table7.loc["Ortalama ± SD"] = [""] + [f"{seed_df[m].mean():.3f} ± {seed_df[m].std():.3f}" for m in METRICS]
table7

## Bölüm B3 — Confusion matrix, ROC ve Precision–Recall eğrileri

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
for ax, (name, label) in zip(axes, [("MLP", f"MLP {SELECTION['selected_candidate']} (seed {PRIMARY})"),
                                    ("LogReg", "Logistic Regression")]):
    m = test_metrics.loc[name]
    cm = np.array([[m["TN"], m["FP"]], [m["FN"], m["TP"]]], dtype=int)
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=cm.max() * 1.15)
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{v}\n(%{100 * v / cm[i].sum():.1f})", ha="center", va="center", fontsize=12,
                color="white" if v > cm.max() * 0.6 else "#0b0b0b")
    ax.set_xticks([0, 1], ["tahmin: no", "tahmin: yes"])
    ax.set_yticks([0, 1], ["gerçek: no", "gerçek: yes"])
    ax.set_title(f"{label}\nSens {m['Sensitivity']:.3f} · Spec {m['Specificity']:.3f}", fontsize=10.5)
    ax.grid(False)
fig.suptitle(f"Bağımsız test confusion matrix (n={len(y_true)}, eşik {THR}) — yüzdeler satır bazında",
             fontweight="bold", y=1.03)
fig.tight_layout()
savefig(fig, "test_confusion_matrices")
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6))
for name, key in [("MLP", "MLP_manual"), ("LogReg", "LogReg")]:
    fpr, tpr, _ = roc_curve(y_true, P[name])
    prec, rec, _ = precision_recall_curve(y_true, P[name])
    lab = f"MLP {SELECTION['selected_candidate']}" if name == "MLP" else "Logistic Regression"
    ax1.plot(fpr, tpr, lw=2, color=yc.METHOD_COLORS[key], label=f"{lab} (ROC-AUC = {test_metrics.loc[name, 'ROC_AUC']:.3f})")
    ax2.step(rec, prec, where="post", lw=2, color=yc.METHOD_COLORS[key], label=f"{lab} (AP = {test_metrics.loc[name, 'AP']:.3f})")
ax1.plot([0, 1], [0, 1], ls=":", color=yc.MUTED, label="rastgele (0.500)")
ax2.axhline(y_true.mean(), ls=":", color=yc.MUTED, label=f"pozitif oran ({y_true.mean():.3f})")
ax1.set(xlabel="False Positive Rate (1 − Specificity)", ylabel="True Positive Rate (Sensitivity)", title="ROC eğrisi")
ax2.set(xlabel="Recall (Sensitivity)", ylabel="Precision", title="Precision–Recall eğrisi", ylim=(0, 1.02))
ax1.legend(fontsize=8.5, loc="lower right")
ax2.legend(fontsize=8.5, loc="upper right")
fig.suptitle("Bağımsız test: ROC ve Precision–Recall eğrileri", fontweight="bold", y=1.01)
fig.tight_layout()
savefig(fig, "test_roc_pr")
plt.show()

## Bölüm B4 — Hata analizi (FP / FN)

Birincil MLP'nin test tahminleri dört gruba ayrılır (TP, FN, FP, TN) ve gruplar seçilmiş özelliklerle betimsel olarak karşılaştırılır. Bu analiz yalnızca yorum içindir; modele veya eşiğe geri besleme yapılmaz.

In [ ]:
pred = (P["MLP"] >= THR).astype(int)
group = np.select([(y_true == 1) & (pred == 1), (y_true == 1) & (pred == 0), (y_true == 0) & (pred == 1)],
                  ["TP", "FN", "FP"], "TN")
err = X_test.assign(grup=group, p_mlp=P["MLP"])
error_table = err.groupby("grup").agg(
    n=("p_mlp", "size"), p_ortalama=("p_mlp", "mean"), p_medyan=("p_mlp", "median"),
    euribor3m_ort=("euribor3m", "mean"), nr_employed_ort=("nr.employed", "mean"),
    önceki_başarı_yüzde=("poutcome", lambda s: 100 * (s == "success").mean()),
    pdays_kayıtlı_yüzde=("pdays", lambda s: 100 * (s != 999).mean()),
    cellular_yüzde=("contact", lambda s: 100 * (s == "cellular").mean()),
    yaş_ort=("age", "mean"), campaign_ort=("campaign", "mean")).reindex(["TP", "FN", "FP", "TN"])
error_table.to_csv(RES_DIR / "error_analysis.csv")

fig, ax = plt.subplots(figsize=(8, 3.6))
bins = np.linspace(0, 1, 41)
for cls, col, lab in [(0, yc.METHOD_COLORS["MLP_manual"], "gerçek: no"), (1, yc.METHOD_COLORS["LogReg"], "gerçek: yes")]:
    ax.hist(P["MLP"][y_true == cls], bins=bins, color=col, alpha=0.75, label=f"{lab} (n={(y_true == cls).sum()})")
ax.axvline(THR, color=yc.INK, ls="--", lw=1)
ax.text(THR, ax.get_ylim()[1] * 0.95, f" eşik {THR}", fontsize=9, va="top")
ax.set_yscale("log")
ax.set(xlabel="Tahmin edilen P(y = yes)", ylabel="gözlem sayısı (log)",
       title=f"Birincil MLP test olasılıklarının sınıfa göre dağılımı")
ax.legend(fontsize=9)
savefig(fig, "test_probability_distribution")
plt.show()
error_table.round(3)

## Bölüm B5 — Kalibrasyon (isteğe bağlı ek analiz)

Brier skoru Tablo 6'dadır. Kalibrasyon eğrisi (10 eşit genişlikli kutu) olasılıkların gerçekleşen oranlarla uyumunu gösterir; test verisiyle herhangi bir kalibrasyon ayarı yapılmaz.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.4))
for name, key in [("MLP", "MLP_manual"), ("LogReg", "LogReg")]:
    frac, mean_p = calibration_curve(y_true, P[name], n_bins=10, strategy="uniform")
    ax.plot(mean_p, frac, "o-", ms=6, lw=2, color=yc.METHOD_COLORS[key],
            label=f"{'MLP ' + SELECTION['selected_candidate'] if name == 'MLP' else 'Logistic Regression'} "
                  f"(Brier {test_metrics.loc[name, 'Brier']:.3f})")
ax.plot([0, 1], [0, 1], ls=":", color=yc.MUTED, label="mükemmel kalibrasyon")
ax.set(xlabel="Ortalama tahmin edilen olasılık", ylabel="Gerçekleşen pozitif oranı", title="Kalibrasyon eğrisi (test)")
ax.legend(fontsize=8.5)
savefig(fig, "test_calibration")
plt.show()

## Bölüm B6 — Aşamalar arası karşılaştırma

Outer CV (arama prosedürünün genellemesi), development iç CV (nihai seçim) ve bağımsız test (kilitli model) farklı soruları yanıtlar; aynı tabloda gösterilir ancak ayrı yorumlanır (yönerge 13.3).

In [ ]:
outer_sum = pd.read_csv(RES_DIR / "outer_cv_summary.csv", header=[0, 1], index_col=0)
dev_ref = pd.read_csv(RES_DIR / "dev_selection_reference_models.csv")
stages = pd.DataFrame({
    "Outer CV (MLP manuel arama, n=3)": [f"{outer_sum.loc['MLP_manual', ('AP', 'mean')]:.3f} ± {outer_sum.loc['MLP_manual', ('AP', 'std')]:.3f}",
                                         f"{outer_sum.loc['LogReg', ('AP', 'mean')]:.3f} ± {outer_sum.loc['LogReg', ('AP', 'std')]:.3f}"],
    "Development iç CV (n=3)": [f"{SELECTION['dev_inner_AP_mean']:.3f} ± {SELECTION['dev_inner_AP_sd']:.3f}",
                                f"{dev_ref.loc[dev_ref.method == 'LogReg', 'AP'].mean():.3f} ± {dev_ref.loc[dev_ref.method == 'LogReg', 'AP'].std():.3f}"],
    "Bağımsız test (nokta [%95 CI])": [
        f"{test_metrics.loc['MLP', 'AP']:.3f} " + ci.query("model == 'MLP' and metrik == 'AP'").pipe(lambda d: f"[{d.CI_alt.iloc[0]:.3f}, {d.CI_üst.iloc[0]:.3f}]"),
        f"{test_metrics.loc['LogReg', 'AP']:.3f} " + ci.query("model == 'LogReg' and metrik == 'AP'").pipe(lambda d: f"[{d.CI_alt.iloc[0]:.3f}, {d.CI_üst.iloc[0]:.3f}]")],
}, index=[f"MLP ({SELECTION['selected_candidate']})", "Logistic Regression"])
stages.to_csv(RES_DIR / "stage_comparison_ap.csv")
stages

## Rapor şeması — Veri bölünmesi ve nested CV akışı (Şekil 1)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8.6))
ax.set_xlim(0, 13)
ax.set_ylim(0, 10)
ax.axis("off")


def box(x, yb, w, h, text, fc="#ffffff", ec="#8f8d86", fs=9.5, bold=False, ls="-"):
    ax.add_patch(FancyBboxPatch((x, yb), w, h, boxstyle="round,pad=0.04", fc=fc, ec=ec, lw=1.2, ls=ls))
    ax.text(x + w / 2, yb + h / 2, text, ha="center", va="center", fontsize=fs, fontweight="bold" if bold else "normal")


def arrow(x1, y1, x2, y2, text="", dx=0.1):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color=yc.INK, lw=1.3))
    if text:
        ax.text((x1 + x2) / 2 + dx, (y1 + y2) / 2, text, fontsize=8.5, color=yc.INK, va="center")


BLUE, ORANGE, GRAY, GREEN = "#e3eefc", "#fde5d8", "#f0efec", "#ddf3ea"
box(3.0, 9.15, 7.0, 0.7, "Ham veri: bank-additional.csv (4.119 gözlem, 20 özellik) — 'duration' çıkarıldı", GRAY, bold=True)
arrow(5.0, 9.15, 3.6, 8.35)
ax.text(7.0, 8.55, "StratifiedGroupKFold(5): ilk kat → test, kalan 4 kat → development", ha="center", fontsize=8.5, color=yc.INK)
arrow(8.0, 9.15, 10.3, 8.35)
box(0.3, 7.6, 6.6, 0.75, "DEVELOPMENT %80 — 3.295 gözlem (357 pozitif)\nEDA, nested CV, manuel/otomatik arama, nihai eğitim", BLUE, bold=True)
box(8.3, 7.6, 4.4, 0.75, "BAĞIMSIZ TEST %20 — 824 gözlem (94 pozitif)\nkilitli: hiçbir fit / seçim / eşik kararı yok", ORANGE, bold=True)

box(0.3, 2.95, 6.6, 4.35, "", "#fbfbfa", ls="--")
ax.text(0.45, 7.05, "Outer StratifiedKFold(3) — arama prosedürlerinin genellemesini ölçer", fontsize=9.5, fontweight="bold")
arrow(3.6, 7.6, 3.6, 7.3)
box(0.5, 3.55, 4.4, 3.2, "", "#ffffff", ls="--")
ax.text(0.6, 6.68, "Outer Train (≈2.197) → Inner StratifiedKFold(3)\n6 manuel (M1–M6) + 6 otomatik (R1–R6) aday", fontsize=8.8, va="top")
box(0.7, 4.75, 2.3, 1.45, "", "#ffffff")
ax.text(1.85, 6.0, "Inner Train (≈1.465)", ha="center", va="center", fontsize=9)
box(0.8, 4.85, 1.0, 0.95, "fit %85\nön işleme fit\n+ ağırlıklar", BLUE, fs=7.8)
box(1.9, 4.85, 1.0, 0.95, "e-stop %15\nBCE → ES\n+ LR scheduler", GREEN, fs=7.8)
box(3.15, 4.75, 1.6, 1.45, "Inner Val\n(≈732)\nAP ile\naday seçimi", "#ffffff", fs=8.5)
box(0.7, 3.7, 4.05, 0.75, "Seçilen aday → outer train'de yeniden eğitim\n(manuel ve otomatik arama ayrı ayrı)", "#ffffff", fs=8.5)
arrow(2.7, 4.75, 2.7, 4.45)
box(5.1, 3.7, 1.65, 3.05, "Outer Eval\n(≈1.098)\n\nyalnızca\nölçüm\n\nMean ± SD\n(n = 3)", "#ffffff", fs=8.8)
arrow(4.75, 4.07, 5.1, 4.07)
ax.text(0.45, 3.22, "Referans modeller (Dummy, LogReg): aynı outer katlarda, yalnızca outer train ile eğitilir.",
        fontsize=8.3, color=yc.INK, va="center")

box(0.3, 1.55, 6.6, 1.05, "Tüm development üzerinde iç CV (3 kat × 12 aday) → önceden tanımlı kural\n"
                         "(en yüksek mean AP) → nihai konfigürasyon", BLUE, fs=9)
arrow(3.6, 2.95, 3.6, 2.6)
box(7.4, 1.55, 2.5, 1.05, "Nihai eğitim (tüm dev.)\n3 seed: 31065*, 32065, 33065\n%15 e-stop alt kümesi", BLUE, fs=8.6)
arrow(6.9, 2.07, 7.4, 2.07)
box(10.3, 1.55, 2.4, 1.05, "Kilitli test tahminleri\n(etiketsiz CSV)\n→ ayrı git commit", ORANGE, fs=8.6)
arrow(9.9, 2.07, 10.3, 2.07)
box(10.3, 0.15, 2.4, 1.0, "Test metrikleri, ROC/PR,\nbootstrap %95 CI,\nseed kararlılığı", ORANGE, fs=8.6)
arrow(11.5, 1.55, 11.5, 1.15)
arrow(10.5, 7.6, 10.5, 2.6, "yalnızca özellikler", dx=0.1)
ax.text(0.3, 0.55, "* birincil (ana) seed, önceden ilan edildi. Tüm ön işleme parametreleri yalnızca ilgili 'fit' alt kümesinde öğrenilir.",
        fontsize=8.3, color=yc.INK)
savefig(fig, "nested_cv_flow")
plt.show()

## Bu notebook'un çıktıları

| Dosya | İçerik |
|---|---|
| `results/final_training_runs.csv`, `results/final_training_history.csv` | Nihai eğitimlerin (3 seed) özetleri ve epoch günlükleri |
| `results/final_hyperparameters.json` | Nihai konfigürasyon, mimari, seed'ler, epoch'lar, öğrenilmiş ön işleme istatistikleri, eşik, ortam |
| `results/models/` | 3 MLP ağırlığı (`.pt`), 3 ön işleyici ve Logistic Regression (`.pkl`) |
| `results/test_predictions_locked.csv` | Etiketsiz kilitli test tahminleri (ayrı commit) |
| `results/test_metrics.csv`, `results/bootstrap_ci.csv` | Tablo 6 — test metrikleri ve bootstrap %95 CI |
| `results/seed_stability_results.csv` | Tablo 7 — üç seed'in test performansı |
| `results/error_analysis.csv`, `results/stage_comparison_ap.csv` | Hata analizi ve aşamalar arası karşılaştırma |
| `figures/final_training_curves.png`, `figures/mlp_architecture.png`, `figures/nested_cv_flow.png` | Nihai eğitim eğrileri, mimari ve akış şemaları |
| `figures/test_*.png` | Confusion matrix, ROC/PR, olasılık dağılımı, kalibrasyon |